In [54]:
using SpeedyWeather, CairoMakie, GLMakie

In [55]:
spectral_grid = SpectralGrid()

SpectralGrid{Spectrum{...}, OctahedralGaussianGrid{...}}
├ Number format: Float32
├ Spectral:      T31 LowerTriangularMatrix
├ Grid:          48-ring OctahedralGaussianGrid, 3168 grid points
├ Resolution:    3.61°, 401km (at 6371km radius)
├ Vertical:      8-layer atmosphere, 2-layer land
└ Architecture:  CPU using Array

In [56]:
model = PrimitiveWetModel(spectral_grid)
simulation = initialize!(model)

Simulation{PrimitiveWetModel}
├ prognostic_variables::PrognosticVariables{...}
├ diagnostic_variables::DiagnosticVariables{...}
└ model::PrimitiveWetModel{...}

## Output variables

In [57]:
model.output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ humid: specific humidity [kg/kg]
 ├ temp: temperature [degC]
 ├ u: zonal wind [m/s]
 ├ mslp: mean sea-level pressure [hPa]
 └ vor: relative vorticity [s^-1]

In [58]:
add!(model, SpeedyWeather.RadiationOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ humid: specific humidity [kg/kg]
 ├ olr: Outgoing longwave radiation [W/m^2]
 └ lru: Surfa

In [59]:
add!(model, SpeedyWeather.SurfaceFluxesOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ slf: Surface latent heat flux (positive up) [W/m^2]
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ shf: Surface sensible heat flux (po

In [60]:
simulation.diagnostic_variables.physics.sensible_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 ⋮
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0

In [61]:
simulation.diagnostic_variables.physics.surface_latent_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 ⋮
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0

In [62]:
# run!(simulation, period=Day(30))
run!(simulation, period=Day(365))

In [63]:
heatmap(simulation.diagnostic_variables.physics.sensible_heat_flux)

In [64]:
simulation.prognostic_variables.clock

Clock
├ time::DateTime = 2000-12-31T00:00:00
├ start::DateTime = 2000-01-01T00:00:00
├ period::Second = 31536000 seconds
├ timestep_counter::Int64 = 13140
├ n_timesteps::Int64 = 13140
└ Δt::Millisecond = 2400000 milliseconds

In [ ]:
function calc_global_sum(field, model)
    a00 = real(transform(field)[1])
    mean_per_m2 = a00 / model.spectral_transform.norm_sphere
    total_W = mean_per_m2 * (4*pi*model.planet.radius^2)
    return total_W
end


calc_global_sum (generic function with 1 method)

In [65]:
function calc_global_mean(field, model)
    a00 = real(transform(field)[1])
    return a00 / model.spectral_transform.norm_sphere    
end
# mean_per_m2 = a00 / model.spectral_transform.norm_sphere

calc_global_mean (generic function with 1 method)

In [66]:
mean_SHF = calc_global_mean(simulation.diagnostic_variables.physics.sensible_heat_flux, model)

27.636118f0

In [67]:
mean_LHF = calc_global_mean(simulation.diagnostic_variables.physics.surface_latent_heat_flux, model)

43.161327f0

In [68]:
simulation.diagnostic_variables.physics

PhysicsVariables
├ grid: OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}
├ ocean: DynamicsVariablesOcean{Float32, Array, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ land: DynamicsVariablesLand{Float32, Array, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ rain_large_scale: 3168-element, 48-ring Field{Float32}
├ rain_convection: 3168-element, 48-ring Field{Float32}
├ snow_large_scale: 3168-element, 48-ring Field{Float32}
├ snow_convection: 3168-element, 48-ring Field{Float32}
├ total_precipitation_rate: 3168-element, 48-ring Field{Float32}
├ cloud_top: 3168-element, 48-ring Field{Float32}

In [73]:
function calc_trenberth_variables(simulation, model; SumFlag::Bool=false)
    # this is a function to calculate the variables we need to plot the Trenberth diagram.
    # simulation -> the SpeedyWeather simulation output.
    # model -> the SpeedyWeather model structure.
    # SumFlag -> false for clculating using the area mean of the fluxes [W/m^2] and true for calculating for the global sum [W]. 

        # put all fields in a Dict
    fields = Dict(
        :LHF   => simulation.diagnostic_variables.physics.surface_latent_heat_flux,
        :SHF   => simulation.diagnostic_variables.physics.sensible_heat_flux,
        :SSRU  => simulation.diagnostic_variables.physics.surface_shortwave_up,
        :SLRU  => simulation.diagnostic_variables.physics.surface_longwave_up,
        :SSRD  => simulation.diagnostic_variables.physics.surface_shortwave_down,
        :SLRD  => simulation.diagnostic_variables.physics.surface_longwave_down,
        :OSR   => simulation.diagnostic_variables.physics.outgoing_shortwave_radiation,
        :OLR   => simulation.diagnostic_variables.physics.outgoing_longwave_radiation,
        :albedo => simulation.diagnostic_variables.physics.albedo
    )

    # identify the relevat fields:
    # LHF = simulation.diagnostic_variables.physics.surface_latent_heat_flux # surface latent heat up
    # SHF = simulation.diagnostic_variables.physics.sensible_heat_flux # surface sensible heat up
    # SSRU = simulation.diagnostic_variables.physics.surface_shortwave_up # surface shortwave up
    # SLRU = simulation.diagnostic_variables.physics.surface_longwave_up # surface longwave up
    # SSRD = simulation.diagnostic_variables.physics.surface_shortwave_down # surface shortwave down
    # SLRD = simulation.diagnostic_variables.physics.surface_longwave_down # surface longwave down
    # OSR = simulation.diagnostic_variables.physics.outgoing_shortwave_radiation # outgoing shortwave radiation (TOA)
    # OLR = simulation.diagnostic_variables.physics.outgoing_longwave_radiation # outgoing longwave radiation (TOA)
    # albedo = simulation.diagnostic_variables.physics.albedo # albedo

    # initialize result container
    results = Dict{Symbol, Float64}()

    # pick the function once
    calcfun = SumFlag ? calc_global_sum : calc_global_mean

    for (name, field) in fields
        try
            results[name] = calcfun(field, model)
        catch err
            # more informative error handling
            @warn "Could not compute $name: $err"
            results[name] = NaN
        end
    end

        # optional derived Trenberth terms (example: ASR, surface net radiation)
    # note: sign conventions may vary in your model; adapt as needed
    # ASR (absorbed shortwave at TOA) = incoming_TOA - reflected_TOA
    # If you only have OSR as outgoing shortwave at TOA, and you know S_in_TOA:
    # results[:ASR] = S_in_global_total - results[:OSR]  # only if you have S_in_TOA
    #
    # Simple surface net (down - up) :
    results[:SW_net_sfc] = results[:SSRD] - results[:SSRU]    # W/m2 or W
    results[:LW_net_sfc] = results[:SLRD] - results[:SLRU]
    results[:surface_net]  = results[:SW_net_sfc] + results[:LW_net_sfc] - results[:LHF] - results[:SHF]

    return results

    return results
end


calc_trenberth_variables (generic function with 2 methods)

In [77]:
R_mean = calc_trenberth_variables(simulation, model; SumFlag=false)  # global means

Dict{Symbol, Float64} with 12 entries:
  :LW_net_sfc  => -237.881
  :OLR         => 301.847
  :SSRD        => 341.257
  :SLRD        => 0.0
  :SHF         => 27.6361
  :SSRU        => 27.2284
  :SLRU        => 237.881
  :LHF         => 43.1613
  :albedo      => 0.116091
  :SW_net_sfc  => 314.029
  :OSR         => 27.2284
  :surface_net => 5.35018